# GOZ-Code-Extraktion: Training & Inferenz (Colab)

Dieses Notebook ist für **Google Colab mit T4-GPU** gebaut (QLoRA-Training
braucht eine GPU, die es in der lokalen Entwicklungsumgebung dieses Projekts
nicht gibt). Es deckt zwei Ansätze ab, die später gegeneinander evaluiert
werden (siehe Task 11, Eval-Report):

1. **RAG-Baseline**: Basismodell (`meta-llama/Llama-3.2-3B-Instruct`) +
   BM25/Embedding-Retrieval über die GOZ-Codeliste als Prompt-Kontext.
2. **QLoRA-Finetune**: dasselbe Basismodell, per LoRA auf den generierten
   Trainingsdaten (`data/train.jsonl`) feingetunt, ohne Retrieval-Kontext
   zur Inferenzzeit (Wissen steckt in den LoRA-Gewichten).

Beide Ansätze laufen am Ende über dasselbe Test-Set (`data/test.jsonl`)
und schreiben ihre Vorhersagen nach `results/predictions_rag.jsonl` bzw.
`results/predictions_finetune.jsonl` — das gemeinsame Format pro Zeile ist
`{"text": ..., "expected_codes": [...], "predicted_codes": [...]}`.

**Wichtig:** Für `meta-llama/Llama-3.2-3B-Instruct` muss die Lizenz auf
Hugging Face vorher akzeptiert und ein HF-Token mit Zugriff bereitliegen
(siehe Zelle 2).


## 1. Setup & Daten laden

Installiert alle Abhängigkeiten, loggt sich bei Hugging Face ein (nötig für
den Download des lizenzpflichtigen Llama-3.2-Modells), lädt die Projekt-
Dateien hoch und liest die kuratierten GOZ-Codes sowie Train-/Test-Split.


In [ ]:
!pip install -q transformers peft bitsandbytes accelerate datasets sentence-transformers rank-bm25 pydantic
# Colabs vorinstalliertes torchao (0.10.0) ist zu alt für die von peft
# geforderte Version (verlangt >0.16.0) - peft.PeftModel.from_pretrained
# crasht sonst mit ImportError beim Laden des LoRA-Adapters. Live zweimal
# bestätigt, kein Rate-Pin.
!pip install -q -U torchao


In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # HF-Token mit akzeptierter Llama-3.2-Lizenz eingeben


In [ ]:
# Repo-Dateien hochladen: src/goz_extract/, data/goz_codes.json, data/train.jsonl, data/test.jsonl
from google.colab import files
uploaded = files.upload()  # als .zip hochladen und entpacken, siehe README-Setup-Abschnitt
!unzip -o goz-extract-src.zip -d .
import sys; sys.path.insert(0, "src")


In [ ]:
import json
from pathlib import Path

from goz_extract.schema import GozCode, NoteExample

codes = [GozCode(**c) for c in json.loads(Path("data/goz_codes.json").read_text(encoding="utf-8"))]
train = [NoteExample.model_validate_json(l) for l in Path("data/train.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
test = [NoteExample.model_validate_json(l) for l in Path("data/test.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
valid_codes = {c.goz_nr for c in codes}
print(len(codes), len(train), len(test))


**Erwartete Ausgabe:** `10 325 81` (10 kuratierte Kern-Codes, nach zusätzlicher Datengenerierung fürs Kern-Set am 2026-07-28 - siehe HANDOVER.md).


## 2. RAG-Baseline

Baut BM25- und Embedding-Index über die GOZ-Codeliste auf (kombiniert per
Reciprocal Rank Fusion in `retrieve_candidates`, siehe Task 6), lädt das
unveränderte Basismodell und lässt es pro Test-Notiz mit den Top-5-
Retrieval-Kandidaten als Prompt-Kontext antworten (top_n=5 statt 12 seit
der Code-Reduktion auf 10 Kern-Codes - mit top_n=12 wären ohnehin immer
alle 10 Codes als "Kandidaten" gezeigt worden, das Retrieval hätte gar
nichts mehr genarrowed).


In [ ]:
from sentence_transformers import SentenceTransformer

from goz_extract.retrieval import BM25Index, EmbeddingIndex, retrieve_candidates

embed_model = SentenceTransformer("intfloat/multilingual-e5-base")


# intfloat/multilingual-e5-base ist asymmetrisch trainiert: Korpus-Texte
# brauchen "passage: ", Suchanfragen "query: " - dieselbe Praefixierung fuer
# beide Seiten wuerde die Embedding-Retrieval-Qualitaet verschlechtern.
def encode_passages(texts):
    return embed_model.encode([f"passage: {t}" for t in texts], normalize_embeddings=False)


def encode_query(texts):
    return embed_model.encode([f"query: {t}" for t in texts], normalize_embeddings=False)


bm25_index = BM25Index(codes)
embedding_index = EmbeddingIndex(codes, encode_fn=encode_passages, encode_query_fn=encode_query)
code_by_nr = {c.goz_nr: c for c in codes}


In [ ]:
from goz_extract.inference import generate_codes, load_model

base_model, tokenizer = load_model("meta-llama/Llama-3.2-3B-Instruct", device_map={"": 0})

rag_predictions = []
for example in test:
    candidate_codes = [code_by_nr[nr] for nr in retrieve_candidates(example.text, bm25_index, embedding_index, top_n=5)]
    predicted = generate_codes(base_model, tokenizer, example.text, valid_codes, candidates=candidate_codes)
    rag_predictions.append({"text": example.text, "expected_codes": example.expected_codes, "predicted_codes": predicted})

print(rag_predictions[0])


**Erwartetes Verhalten:** läuft ohne Fehler über alle Test-Notizen durch
(Dauer: ca. 1–3 Minuten auf T4 für ~80–100 Notizen), `predicted_codes` in
jedem Eintrag ist eine Liste gültiger GOZ-Ziffern.


In [ ]:
import json
Path("results").mkdir(exist_ok=True)
with open("results/predictions_rag.jsonl", "w", encoding="utf-8") as f:
    for row in rag_predictions:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


## 3. LoRA-Training (QLoRA)

Baut aus den Trainingsbeispielen Chat-formatierte Texte (Prompt ohne
Retrieval-Kandidaten — das Finetune soll die Codes aus den Gewichten lernen,
nicht aus Kontext), lädt das Basismodell zusätzlich 4-bit-quantisiert für
QLoRA und trainiert einen LoRA-Adapter darauf.


In [ ]:
import json

from datasets import Dataset
from goz_extract.prompting import build_extraction_prompt

def to_chat_example(example):
    prompt = build_extraction_prompt(example.text, candidates=None)
    completion = json.dumps(example.expected_codes, ensure_ascii=False)
    messages = [{"role": "user", "content": prompt}, {"role": "assistant", "content": completion}]
    return {"messages": messages}

train_dataset = Dataset.from_list([to_chat_example(e) for e in train])


In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

# TRLs eingebaute Completion-Only-Loss-Mechanismen sind hier zweimal live
# gescheitert: DataCollatorForCompletionOnlyLM existiert in trl>=1.x nicht
# mehr, und der Nachfolger SFTConfig(assistant_only_loss=True) braucht
# {% generation %}-Marker im Chat-Template, die Llamas Standard-Template
# nicht hat (TRL kann das nicht automatisch patchen). Deshalb hier bewusst
# ohne SFTTrainer: Labels werden manuell maskiert (Prompt-Teil = -100,
# nur die Antwort trägt zum Loss bei) und mit dem einfacheren, stabileren
# transformers.Trainer trainiert - unabhängig von TRLs sich änderndem
# High-Level-API.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
train_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct", quantization_config=bnb_config, device_map="auto"
)
train_model = prepare_model_for_kbit_training(train_model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
train_model = get_peft_model(train_model, lora_config)


def tokenize_with_completion_mask(example):
    messages = example["messages"]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    # Prompt bis einschließlich der "assistant"-Kopfzeile rendern (aber ohne
    # Antwortinhalt) - die Länge davon markiert, bis wohin maskiert wird.
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1], tokenize=False, add_generation_prompt=True
    )

    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    prompt_len = min(len(prompt_ids), len(full_ids))

    labels = full_ids.copy()
    for i in range(prompt_len):
        labels[i] = -100

    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}


tokenized_train = train_dataset.map(
    tokenize_with_completion_mask, remove_columns=train_dataset.column_names
)


def collate_fn(batch):
    max_len = max(len(ex["input_ids"]) for ex in batch)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    input_ids, attention_mask, labels = [], [], []
    for ex in batch:
        pad_len = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [pad_id] * pad_len)
        attention_mask.append(ex["attention_mask"] + [0] * pad_len)
        labels.append(ex["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }


In [ ]:
# Sanity-Check vor dem eigentlichen Training: BPE-Tokenisierung ist an
# Konkatenationsgrenzen nicht garantiert praefix-stabil (Merges koennen sich
# je nach Folgezeichen unterscheiden) - tokenize_with_completion_mask oben
# verlaesst sich aber genau darauf (prompt_ids als exaktes Praefix von
# full_ids). Hier wird fuer eine Stichprobe verifiziert, dass die Maskierung
# tatsaechlich an der richtigen Stelle sitzt, bevor GPU-Zeit in einen vollen
# Lauf investiert wird - sonst koennte unbemerkt wieder ein Stueck
# Prompt-Boilerplate ins Lernsignal rutschen (der Fehler hinter dem
# urspruenglichen Mode Collapse).
import random

sample_size = min(10, len(tokenized_train))
for i in random.sample(range(len(tokenized_train)), sample_size):
    row = tokenized_train[i]
    kept_ids = [t for t, l in zip(row["input_ids"], row["labels"]) if l != -100]
    decoded = tokenizer.decode(kept_ids, skip_special_tokens=True).strip()
    expected = json.dumps(train[i].expected_codes, ensure_ascii=False)
    assert decoded == expected, (
        f"Completion-Masking-Grenze falsch bei Beispiel {i}: "
        f"decodiert {decoded!r} != erwartet {expected!r}"
    )

print(f"Completion-Masking verifiziert an {sample_size} Stichproben: Labels entsprechen exakt den erwarteten Codes.")


**Falls die Assertion hier fehlschlaegt:** nicht einfach den Assert entfernen und weitermachen - das bedeutet, dass die Completion-Maskierung an der falschen Stelle sitzt und der Trainingslauf denselben Mode-Collapse-Bug reproduzieren wuerde, den Anlauf 3 eigentlich beheben sollte. Stattdessen `tokenize_with_completion_mask` debuggen (z.B. `prompt_ids` und die ersten Tokens von `full_ids` nebeneinander decodieren).


In [ ]:
training_args = TrainingArguments(
    output_dir="adapters/goz-extract-llama32-3b",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,  # sonst je Checkpoint ~100MB (Adapter + Optimizer-State)
    # bei 10 Epochen statt 3 - nur den neuesten behalten, spart Zeit/Speicher
    # beim Download.
)

trainer = Trainer(
    model=train_model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=collate_fn,
)
trainer.train()
train_model.save_pretrained("adapters/goz-extract-llama32-3b")
tokenizer.save_pretrained("adapters/goz-extract-llama32-3b")


Bevor das feingetunte Modell geladen wird: `base_model` (RAG-Baseline,
bf16/fp16), `train_model` (4-bit-QLoRA-Trainingskopie) und `trainer`
liegen zu diesem Zeitpunkt noch im GPU-Speicher. Zusammen mit einer
dritten Modellkopie für die Finetune-Inferenz würde das auf einer T4
(~15GB VRAM) sehr wahrscheinlich zu einem CUDA-Out-of-Memory führen.
Die folgende Zelle gibt den Speicher der nicht mehr gebrauchten
Objekte frei, bevor `finetuned_model` geladen wird.


In [ ]:
import gc, torch

for _name in ["base_model", "train_model", "trainer", "tokenized_train"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()


In [ ]:
finetuned_model, ft_tokenizer = load_model(
    "meta-llama/Llama-3.2-3B-Instruct",
    adapter_path="adapters/goz-extract-llama32-3b",
    device_map={"": 0},  # kein Auto-Offload - kollidiert live mit peft's Adapter-Ladelogik
)

finetune_predictions = []
for example in test:
    predicted = generate_codes(finetuned_model, ft_tokenizer, example.text, valid_codes, candidates=None)
    finetune_predictions.append({"text": example.text, "expected_codes": example.expected_codes, "predicted_codes": predicted})

with open("results/predictions_finetune.jsonl", "w", encoding="utf-8") as f:
    for row in finetune_predictions:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(finetune_predictions[0])


**Erwartetes Verhalten:** läuft über alle Test-Notizen durch,
`predicted_codes` enthält gültige GOZ-Ziffern.


## 4b. Checker-Node (Graph-Ausbau)

Der erste Graph-Versuch (Aggregator + Verifier, `graph_merge.py`) brachte
**nichts**: Die Merge-Regel ist algebraisch identisch zum Finetune —
Exact Match 0.38 bei 95 % Prüfquote. Die Diagnose
(`scripts/analyze_merge_headroom.py`) zeigt aber, dass die erwarteten Codes
in **65 von 81 Notizen (0.80)** vollständig in der Vereinigung beider Pfade
stecken. Der Hebel liegt also im *Auswählen*, nicht im *Verrechnen*.

Diese Zelle lässt genau das laufen: Kandidaten beider Pfade poolen, das
**unveränderte Basismodell** (nicht das Finetune — sonst wäre der Vergleich
schief) wählt daraus aus. Voraussetzung: Die Zellen für RAG- und
Finetune-Vorhersagen oben sind durchgelaufen, `rag_predictions` und
`finetune_predictions` liegen im Speicher.

Kein Retrieval, kein Adapter: Der Kandidatenpool *ist* das Domänenwissen.


In [ ]:
from goz_extract.checker import checker_predict

# Basismodell ohne Adapter - dasselbe, das auch die RAG-Baseline benutzt.
# Falls oben mit dem Finetune weitergearbeitet wurde, hier neu laden:
#   base_model, tokenizer = load_model("meta-llama/Llama-3.2-3B-Instruct", device_map={"": 0})

code_by_nr = {c.goz_nr: c for c in codes}


def select_with_base_model(note_text, candidates):
    """Auswahlfunktion des Checkers - dieselbe Mechanik wie im RAG-Pfad,
    nur mit den gepoolten Modell-Kandidaten statt Retrieval-Treffern."""
    return generate_codes(
        base_model, tokenizer, note_text, valid_codes, candidates=candidates
    )


rag_by_text = {row["text"]: row["predicted_codes"] for row in rag_predictions}
ft_by_text = {row["text"]: row["predicted_codes"] for row in finetune_predictions}

checker_predictions = []
for example in test:
    result = checker_predict(
        example.text,
        rag_by_text[example.text],
        ft_by_text[example.text],
        code_by_nr,
        select_with_base_model,
    )
    checker_predictions.append({
        "text": example.text,
        "expected_codes": example.expected_codes,
        "predicted_codes": result.predicted_codes,
        "pooled": result.sources["gepoolt"],
        "needs_review": result.needs_review,
    })

print(checker_predictions[0])
print(f"needs_review: {sum(r['needs_review'] for r in checker_predictions)}/{len(checker_predictions)}")

**Erwartetes Verhalten:** läuft über alle Test-Notizen, `predicted_codes`
ist immer eine Teilmenge von `pooled`. Die Obergrenze liegt bei Exact Match
0.80 (perfekte Auswahl) — alles deutlich über 0.38 ist ein echter Gewinn
gegenüber dem Finetune, alles darunter heißt: Die Auswahlaufgabe ist für
dieses Modell zu schwer, und der Graph bleibt hier die falsche Antwort.
Beides ist ein Ergebnis, beides gehört ins README.

In [ ]:
import json
from pathlib import Path

Path("results").mkdir(exist_ok=True)
with open("results/predictions_graph_checker.jsonl", "w", encoding="utf-8") as f:
    for row in checker_predictions:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

from google.colab import files
files.download("results/predictions_graph_checker.jsonl")

Die heruntergeladene Datei nach `results/` ins Repo legen, dann lokal:

```bash
python scripts/run_checker_eval.py --results-dir results/ --backend jsonl \
    --checker-predictions results/predictions_graph_checker.jsonl
```

Das ist die **faire** Zeile für die Ergebnistabelle: gleiches Basismodell wie
RAG-Baseline und Finetune, nur anders verdrahtet.

## 5. Artefakte herunterladen & committen (manueller Schritt)

Die folgenden zwei Schritte passieren **außerhalb** dieses Notebooks, von
einem Menschen, nachdem die Zellen oben auf Colab erfolgreich durchgelaufen
sind — sie sind hier nur dokumentiert, nicht als ausführbare Zellen, weil
sie lokalen Datei-/Git-Zugriff brauchen, den Colab nicht hat:

1. **Artefakte herunterladen:** `results/predictions_rag.jsonl`,
   `results/predictions_finetune.jsonl` und den Ordner
   `adapters/goz-extract-llama32-3b/` aus Colab herunterladen
   (Dateibrowser oder `files.download(...)`) und lokal nach
   `goz-finetune-vs-rag/results/` bzw. `goz-finetune-vs-rag/adapters/`
   legen.
2. **Committen:**
   ```bash
   git add notebooks/train_and_infer.ipynb results/predictions_rag.jsonl results/predictions_finetune.jsonl
   git commit -m "Add Colab notebook for QLoRA training and RAG-baseline/finetune inference over the test set"
   ```
   (LoRA-Adapter-Gewichte unter `adapters/` sind groß — vor dem Commit
   prüfen, ob sie stattdessen per `.gitignore` ausgeschlossen und separat
   z. B. auf Hugging Face Hub hochgeladen werden sollen; im README
   verlinken.)


In [ ]:
!zip -r adapter.zip adapters/goz-extract-llama32-3b
from google.colab import files
files.download("adapter.zip")
